# Simón dice: emociones

El juego te pide una emoción y tienes unos segundos para ponerla. Si nuestro modelo la ve, sumas un punto y la siguiente ronda va más rápida. Si se acaba el tiempo pierdes una vida. Con 3 vidas perdidas se acaba.

Usamos:
- **MediaPipe** para encontrar la cara y recortarla.
- **Nuestro modelo** (`modelo_emociones.pt`) para saber qué emoción pone.

Teclas: **espacio** para volver a jugar, **Esc** para salir.

## 1. Cargar el modelo y MediaPipe

El modelo se crea igual que en el entrenamiento y le cargamos los pesos que guardamos. La transformación tiene que ser la misma que al entrenar.

In [7]:
import cv2
import time
import random
import torch
import torch.nn as nn
import mediapipe as mp
from PIL import Image
from torchvision import models, transforms

emociones = ["Anger", "Contempt", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"

modelo = models.resnet18()
modelo.fc = nn.Linear(512, 8)
modelo.load_state_dict(torch.load("modelo_emociones.pt", map_location=dispositivo))
modelo = modelo.to(dispositivo)
modelo.eval()

transformacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

opciones = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=mp.tasks.vision.RunningMode.IMAGE)
detector = mp.tasks.vision.FaceLandmarker.create_from_options(opciones)

## 2. Función para sacar la emoción de un frame

1. MediaPipe nos da los puntos de la cara.
2. Buscamos el punto más a la izquierda, derecha, arriba y abajo → eso es el rectángulo de la cara.
3. Le añadimos un margen del 25%, porque las fotos de AffectNet no están recortadas justo a la cara (se ve el pelo y la barbilla). Sin el margen el modelo ve las caras distintas a como las aprendió y falla más.
4. Recortamos la cara y se la pasamos al modelo.
5. `softmax` convierte la salida en probabilidades, y nos quedamos con la más alta.

In [8]:
def sacar_emocion(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultado = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    # si no hay cara no devolvemos nada
    if len(resultado.face_landmarks) == 0:
        return None, 0, None

    alto = frame.shape[0]
    ancho = frame.shape[1]

    # rectángulo de la cara
    x_min = ancho
    x_max = 0
    y_min = alto
    y_max = 0
    for punto in resultado.face_landmarks[0]:
        x = int(punto.x * ancho)
        y = int(punto.y * alto)
        x_min = min(x_min, x)
        x_max = max(x_max, x)
        y_min = min(y_min, y)
        y_max = max(y_max, y)

    # le damos un margen del 25% para no perder cara
    margen = int((x_max - x_min) * 0.25)
    x_min = max(x_min - margen, 0)
    y_min = max(y_min - margen, 0)
    x_max = x_max + margen
    y_max = y_max + margen

    # recortar la cara y pasarla al modelo
    cara = Image.fromarray(rgb[y_min:y_max, x_min:x_max])
    with torch.no_grad():
        salida = modelo(transformacion(cara).unsqueeze(0).to(dispositivo))
    probabilidades = salida.softmax(1)[0]

    emocion = emociones[probabilidades.argmax().item()]
    prob = probabilidades.max().item()
    return emocion, prob, (x_min, y_min, x_max, y_max)

# probamos con una foto
emocion, prob, caja = sacar_emocion(cv2.imread("faces/happy.jpg"))
print(emocion, prob)

Happy 0.999971866607666


## 3. El juego

Variables:
- `puntos`, `vidas`
- `objetivo`: la emoción que pide el juego
- `inicio`: cuándo empezó la ronda, y `tiempo`: segundos que tienes
- `en_espanol`: diccionario para enseñar en pantalla el nombre de cada emoción en español. Ojo: no puede llamarse `emociones`, porque esa es la lista que usa `sacar_emocion` para saber qué emoción es cada salida del modelo.
- `seguidos`: cuántos frames seguidos lleva viendo la emoción buena. Pedimos 8 seguidos para que no cuente un acierto de casualidad.

Cada vez que aciertas, el tiempo baja 0.2 segundos (como mínimo 1.5).

OpenCV no escribe tildes, por eso los textos van sin ellas.

In [9]:
# quitamos Neutral y Contempt, son complejas de detectar
para_jugar = ["Anger", "Disgust", "Fear", "Happy", "Sad", "Surprise"]
# nombre en español para enseñarlo en pantalla (no llamarlo "emociones", que es la lista del modelo)
en_espanol = {"Anger": "ENFADO", "Contempt": "DESPRECIO", "Disgust": "ASCO", "Fear": "MIEDO",
              "Happy": "ALEGRIA", "Neutral": "NEUTRAL", "Sad": "TRISTEZA", "Surprise": "SORPRESA"}

puntos = 0
vidas = 3
tiempo = 5
objetivo = random.choice(para_jugar)
inicio = time.time()
seguidos = 0
mensaje = ""
hora_mensaje = 0

cap = cv2.VideoCapture(0)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)   # efecto espejo

    emocion, prob, caja = sacar_emocion(frame)

    # dibujamos la cara y lo que ve el modelo
    if caja is not None:
        cv2.rectangle(frame, (caja[0], caja[1]), (caja[2], caja[3]), (255, 255, 255), 2)
        texto = "Veo: " + en_espanol[emocion] + " " + str(int(prob * 100)) + "%"
        cv2.putText(frame, texto, (caja[0], caja[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    if vidas > 0:
        if emocion == objetivo and prob > 0.5:
            seguidos = seguidos + 1
        else:
            seguidos = 0

        restante = tiempo - (time.time() - inicio)

        if seguidos >= 8:
            puntos = puntos + 1
            tiempo = max(1.5, tiempo - 0.2)
            objetivo = random.choice(para_jugar)
            inicio = time.time()
            seguidos = 0
            mensaje = "BIEN!"
            hora_mensaje = time.time()
        elif restante <= 0:
            vidas = vidas - 1
            objetivo = random.choice(para_jugar)
            inicio = time.time()
            seguidos = 0
            mensaje = "FALLO!"
            hora_mensaje = time.time()

        # lo que pide el juego
        cv2.putText(frame, "Pon cara de: " + en_espanol[objetivo], (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        cv2.putText(frame, "Tiempo: " + str(round(restante, 1)), (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # el mensaje de bien/fallo
        if time.time() - hora_mensaje < 1:
            if mensaje == "BIEN!":
                color = (0, 255, 0)
            else:
                color = (0, 0, 255)
            cv2.putText(frame, mensaje, (20, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 4)
    else:
        cv2.putText(frame, "FIN DEL JUEGO", (20, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
        cv2.putText(frame, "Pulsa espacio para jugar otra vez", (20, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # marcador
    cv2.putText(frame, "Puntos: " + str(puntos) + "   Vidas: " + str(vidas), (20, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow("Simon dice", frame)

    tecla = cv2.waitKey(1)
    if tecla == 27:
        break
    if tecla == 32:
        puntos = 0
        vidas = 3
        tiempo = 5
        objetivo = random.choice(para_jugar)
        inicio = time.time()
        seguidos = 0

cap.release()
cv2.destroyAllWindows()